---
## What I Built

- FastAPI ML service: /predict, /health, /metrics endpoints
- Docker container for reproducible deployment
- Load testing results

## Latency Results

| Metric | Target | Actual |
|--------|--------|--------|
| p50 | < 20ms | |
| p95 | < 100ms | |
| p99 | < 200ms | |

## What I Learned
- 

In [ ]:
# TODO: Simple load test
# import concurrent.futures, time, statistics
# Run 200 requests concurrently with 10 workers
# Report: mean, p50, p95, p99 latency
# Target: p95 < 100ms

In [ ]:
# TODO: Test all endpoints
# import requests
# BASE = "http://localhost:8000"
#
# Test 1: GET /health
# Test 2: POST /v1/predict with valid data
# Test 3: POST /v1/predict with invalid data (expect 422)
# Test 4: POST /v1/predict/batch with 10 customers
# Test 5: GET /v1/model/info

---
## 4. Test the API

In [ ]:
# TODO: Create Dockerfile
dockerfile = """
FROM python:3.11-slim
WORKDIR /app
# TODO: Copy requirements.txt and install
# TODO: Copy src/ directory
# TODO: Expose port 8000
# TODO: Set CMD to run uvicorn
"""
# with open('Dockerfile', 'w') as f:
#     f.write(dockerfile)

# TODO: Create requirements.txt

# To build: docker build -t churn-api:v1 .
# To run:   docker run -p 8000:8000 churn-api:v1
print("TODO: implement Dockerfile and requirements.txt")

---
## 3. Containerize

In [ ]:
# TODO: Create src/main.py with:
# - Load model from src/model.pkl at startup
# - CustomerFeatures Pydantic model (with validation)
# - POST /v1/predict — single prediction
# - POST /v1/predict/batch — batch predictions (max 500)
# - GET /health — health check
# - GET /metrics — simple metrics (request count, mean latency)
# - GET /v1/model/info — model metadata

# Scaffold:
fastapi_code = """
from fastapi import FastAPI
from pydantic import BaseModel, Field
import pickle, time

app = FastAPI(title="Churn Prediction API v1", version="1.0.0")
MODEL_DATA = {}

@app.on_event("startup")
async def load_model():
    # TODO: Load src/model.pkl
    pass

# TODO: Define CustomerFeatures with Pydantic

@app.get("/health")
async def health():
    # TODO
    pass

@app.post("/v1/predict")
async def predict(customer: CustomerFeatures):  # noqa
    # TODO
    pass
"""
print("TODO: implement src/main.py")

---
## 2. Build the FastAPI App

Create `src/main.py` with all endpoints.

In [ ]:
# Pre-built: train and save model
import pickle, numpy as np, pandas as pd
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from pathlib import Path

np.random.seed(42); n = 8000
tenure = np.random.exponential(24, n).clip(1, 72).astype(int)
monthly = np.random.normal(65, 25, n).clip(20, 150)
total = tenure * monthly + np.random.normal(0, 500, n).clip(0)
contract = np.random.choice([0, 1, 2], n, p=[0.55, 0.25, 0.20])
num_prod = np.random.randint(1, 6, n)
calls = np.random.poisson(2, n)
prob = 1 / (1 + np.exp(-(-2 + 0.05*(monthly-65) - 0.03*tenure + 0.1*calls - 0.5*contract)))
y = (np.random.random(n) < prob).astype(int)

X = pd.DataFrame({'tenure_months': tenure, 'monthly_charges': monthly,
                  'total_charges': total, 'contract_encoded': contract,
                  'num_products': num_prod, 'support_calls': calls})

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
model = GradientBoostingClassifier(n_estimators=150, learning_rate=0.08, max_depth=4, random_state=42)
model.fit(X_train, y_train)
auc = roc_auc_score(y_val, model.predict_proba(X_val)[:,1])
print(f"Model AUC: {auc:.4f}")

model_dir = Path('src')
model_dir.mkdir(exist_ok=True)
with open(model_dir / 'model.pkl', 'wb') as f:
    pickle.dump({'model': model, 'feature_cols': X.columns.tolist(), 'auc': auc, 'version': '1.0.0'}, f)
print("Saved: src/model.pkl")

In [ ]:
import pickle, json, os
import numpy as np, pandas as pd
import warnings; warnings.filterwarnings('ignore')
print("Setup complete")

# P3: Dockerized ML API

**Module:** 3 — MLOps
**Week:** 12

## What You'll Build
A production-grade FastAPI ML service that:
- Serves churn predictions with validation
- Runs in a Docker container
- Has health checks and structured logging
- Can handle concurrent requests